In [2]:
import pandas as pd
import numpy as np 
import os 

In [3]:
df = pd.read_csv("/kaggle/input/datasets/yousrakerrouch/dataset-17cols/irrigation_prediction_17cols.csv")
df.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,Clay,6.14,36.48,0.42,2.17,21.90,31.19,1167.70,4.01,1.97,Wheat,Vegetative,Rabi,Yes,1.98,South,Low
1,Silt,6.41,50.56,0.38,0.23,36.50,26.01,831.28,10.72,16.82,Maize,Flowering,Zaid,Yes,33.56,Central,Medium
2,Sandy,7.71,40.07,1.09,2.18,41.83,76.41,1844.45,7.75,19.03,Cotton,Harvest,Rabi,Yes,34.62,South,Low
3,Clay,5.96,12.75,1.56,0.40,37.22,43.32,306.26,8.90,11.44,Wheat,Sowing,Kharif,Yes,84.03,North,Medium
4,Clay,7.76,18.58,0.95,2.52,22.38,86.44,1875.63,10.39,11.26,Cotton,Sowing,Zaid,No,60.86,South,Medium


## 1. Principe de l'indice de stress hydrique

Le stress hydrique dépend principalement du manque d'eau disponible pour la plante.

Dans ce projet, l'indice `Stress_Index` est construit à partir de plusieurs facteurs :

- Une faible humidité du sol augmente le stress.
- Une température élevée augmente l'évaporation.
- Une faible humidité de l'air favorise la perte d'eau.
- Une faible pluviométrie réduit l'apport naturel en eau.
- Un fort ensoleillement augmente l'évaporation.
- Un vent élevé peut accélérer la perte d'eau.
- Une conductivité électrique élevée peut indiquer un stress lié à la salinité.
- Une faible irrigation précédente peut augmenter le risque de déficit hydrique.
- Le type de sol influence la capacité de rétention d'eau.
- Le type de culture et le stade de croissance influencent les besoins en eau.
- Le paillage réduit le stress en conservant l'humidité du sol.

Cet indice est donc une variable créée manuellement à partir d'une logique agronomique.

## 2. Normalisation des variables pour le calcul

In [4]:
def minmax_normalize(column):
    return (column - column.min()) / (column.max() - column.min())

## Facteurs numériques

Nous transformons chaque variable numérique en un facteur de stress.

Pour certaines variables, une grande valeur augmente le stress, comme la température ou le vent.

Pour d'autres variables, une petite valeur augmente le stress, comme l'humidité du sol, l'humidité de l'air ou la pluie.  
Dans ce cas, nous utilisons `1 - valeur_normalisée`.

In [5]:
soil_dryness = 1 - minmax_normalize(df["Soil_Moisture"])
high_temperature = minmax_normalize(df["Temperature_C"])
low_humidity = 1 - minmax_normalize(df["Humidity"])
low_rainfall = 1 - minmax_normalize(df["Rainfall_mm"])
high_sunlight = minmax_normalize(df["Sunlight_Hours"])
high_wind = minmax_normalize(df["Wind_Speed_kmh"])
high_ec = minmax_normalize(df["Electrical_Conductivity"])
low_previous_irrigation = 1 - minmax_normalize(df["Previous_Irrigation_mm"])

## Facteurs catégoriels

Certaines variables catégorielles ont aussi un effet sur le stress hydrique.

Pour les utiliser dans le calcul, nous leur attribuons des poids numériques.

Ces poids sont définis de manière heuristique, c'est-à-dire selon une logique métier et agronomique.

Par exemple :

- Un sol sableux garde moins bien l'eau, donc il reçoit un poids plus élevé.
- Le paillage aide à conserver l'humidité du sol, donc il réduit l'indice de stress.
- Certaines cultures ont des besoins en eau plus élevés que d'autres.
- Certains stades de croissance sont plus sensibles au manque d'eau.

In [6]:
soil_weight = df["Soil_Type"].map({
    "Sandy": 1.00,
    "Loamy": 0.55,
    "Silt": 0.45,
    "Clay": 0.30
}).fillna(0.50)

stage_weight = df["Crop_Growth_Stage"].map({
    "Sowing": 0.75,
    "Vegetative": 1.00,
    "Flowering": 0.90,
    "Harvest": 0.45
}).fillna(0.60)

crop_weight = df["Crop_Type"].map({
    "Rice": 1.00,
    "Sugarcane": 0.95,
    "Cotton": 0.85,
    "Maize": 0.75,
    "Wheat": 0.65,
    "Soybean": 0.60
}).fillna(0.70)

mulch_factor = df["Mulching_Used"].map({
    "Yes": 0.85,
    "No": 1.00
}).fillna(1.00)

## 3. Calcul du stress de base

Nous calculons d'abord un score de stress de base à partir des facteurs numériques et du type de sol.

Les poids utilisés sont les suivants :

- Sécheresse du sol : 30 %
- Température élevée : 15 %
- Faible pluviométrie : 15 %
- Faible humidité de l'air : 12 %
- Ensoleillement élevé : 8 %
- Vent élevé : 5 %
- Conductivité électrique élevée : 5 %
- Faible irrigation précédente : 5 %
- Type du sol : 5 %

L'humidité du sol reçoit le poids le plus élevé car elle représente l'indicateur le plus direct du stress hydrique.

In [7]:
base_stress = (
    0.30 * soil_dryness +
    0.15 * high_temperature +
    0.12 * low_humidity +
    0.15 * low_rainfall +
    0.08 * high_sunlight +
    0.05 * high_wind +
    0.05 * high_ec +
    0.05 * low_previous_irrigation +
    0.05 * soil_weight
)

## 4. Création de la variable `Stress_Index`

Le score de base est ensuite ajusté selon :

- Le type de culture.
- Le stade de croissance.
- L'utilisation du paillage.

La formule finale permet d'obtenir un indice compris entre 0 et 100.



In [8]:
df["Stress_Index"] = (
    base_stress
    * (0.80 + 0.20 * crop_weight)
    * (0.85 + 0.15 * stage_weight)
    * mulch_factor
    * 100
).clip(0, 100).round(2)

Nous utilisons une multiplication car le type de culture, le stade de croissance et le paillage sont considérés comme des facteurs de correction. Ils ne sont pas ajoutés directement au stress de base, mais modifient ce stress de manière proportionnelle. Ainsi, l'effet de la correction dépend du niveau initial de stress : une même correction aura un impact plus important lorsque le stress de base est élevé, et un impact plus faible lorsque le stress de base est faible.

## 5. Vérification du résultat

Après la création de la nouvelle colonne, nous vérifions que `Stress_Index` a bien été ajouté au dataset.

Nous affichons aussi quelques colonnes importantes pour voir si les valeurs obtenues sont cohérentes avec les conditions climatiques et l'état du sol.

In [9]:
df.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need,Stress_Index
0,Clay,6.14,36.48,0.42,2.17,21.90,31.19,1167.70,4.01,1.97,Wheat,Vegetative,Rabi,Yes,1.98,South,Low,38.53
1,Silt,6.41,50.56,0.38,0.23,36.50,26.01,831.28,10.72,16.82,Maize,Flowering,Zaid,Yes,33.56,Central,Medium,47.40
2,Sandy,7.71,40.07,1.09,2.18,41.83,76.41,1844.45,7.75,19.03,Cotton,Harvest,Rabi,Yes,34.62,South,Low,42.22
3,Clay,5.96,12.75,1.56,0.40,37.22,43.32,306.26,8.90,11.44,Wheat,Sowing,Kharif,Yes,84.03,North,Medium,56.29
4,Clay,7.76,18.58,0.95,2.52,22.38,86.44,1875.63,10.39,11.26,Cotton,Sowing,Zaid,No,60.86,South,Medium,48.94


In [13]:
df[[
    "Soil_Moisture",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Soil_Type",
    "Crop_Type",
    "Mulching_Used",
    "Stress_Index",
    "Irrigation_Need"
]].head(20)

,Soil_Moisture,Temperature_C,Humidity,Rainfall_mm,Soil_Type,Crop_Type,Mulching_Used,Stress_Index,Irrigation_Need
0,36.48,21.90,31.19,1167.70,Clay,Wheat,Yes,38.53,Low
1,50.56,36.50,26.01,831.28,Silt,Maize,Yes,47.40,Medium
2,40.07,41.83,76.41,1844.45,Sandy,Cotton,Yes,42.22,Low
3,12.75,37.22,43.32,306.26,Clay,Wheat,Yes,56.29,Medium
4,18.58,22.38,86.44,1875.63,Clay,Cotton,No,48.94,Medium
5,20.50,33.34,62.51,402.92,Silt,Rice,Yes,55.60,Medium
6,22.70,28.02,58.50,764.95,Sandy,Rice,No,63.31,Medium
7,40.23,35.60,79.10,833.33,Sandy,Wheat,Yes,39.11,Low
8,36.73,16.59,76.88,2476.03,Clay,Maize,Yes,25.55,Low
9,19.58,31.14,74.30,1474.55,Loamy,Rice,Yes,49.93,Medium


In [11]:
df["Stress_Index"].describe()

count    10000.000000
mean        43.130941
std         10.851668
min         11.470000
25%         35.360000
50%         42.740000
75%         50.472500
max         82.210000
Name: Stress_Index, dtype: float64

## 6.Conclusion

Dans ce notebook, nous avons créé une nouvelle variable appelée `Stress_Index`.

Le calcul de l'indice de stress hydrique a été réalisé en deux étapes.

+ Dans un premier temps, un score de stress de base (`base_stress`) a été calculé à partir des facteurs les plus directement liés au déficit hydrique, notamment l'humidité du sol, la température, les précipitations, l'humidité de l'air, l'ensoleillement, la vitesse du vent, la conductivité électrique, l'irrigation précédente et le type de sol. Ce score représente le niveau général de stress causé par les conditions climatiques et pédologiques.

+ Dans un deuxième temps, ce score de base a été ajusté selon des facteurs agronomiques tels que le type de culture, le stade de croissance et l'utilisation du paillage. Cette étape permet d'obtenir un indice final plus réaliste, car deux cultures différentes peuvent réagir différemment aux mêmes conditions climatiques.

L'indice final `Stress_Index` est ensuite multiplié par 100 afin d'obtenir une valeur comprise entre 0 et 100, où une valeur élevée indique un risque plus important de stress hydrique.

L'ajout de cette variable permet d'enrichir le dataset avec une information agronomique plus interprétable.



In [15]:
df.to_csv("/kaggle/working/irrigation_prediction_18cols.csv", index=False)

